In [2]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, save_npz, load_npz
from sklearn.metrics.pairwise import cosine_similarity
import os
import kagglehub

In [3]:
DATA_DIR = os.path.join("..", "data")  # same relative-path pattern as before

train = pd.read_parquet(os.path.join(DATA_DIR, "train_ratings.parquet"))
test = pd.read_parquet(os.path.join(DATA_DIR, "test_ratings.parquet"))

print(train.shape, test.shape)

(118387327, 4) (29596832, 4)


## Baseline 1 (K most popular shows)

In [4]:
# Count positive interactions per anime, using train only
popularity = train[train['is_positive'] == 1].groupby('anime_id').size().sort_values(ascending=False)

#K most popular shows
K = 10
top_k_popular = popularity.head(K).index.tolist()

print(top_k_popular)

#Find the precision and recall rate
def precision_recall_at_k(test_df, recommended_items, k):
    total_relevant = 0
    total_recommended_relevant = 0
    
    for user_id, group in test_df[test_df['is_positive'] == 1].groupby('user_id'):
        actual_positive = set(group['anime_id'])
        recommended = set(recommended_items[:k])
        
        hits_this_user = len(actual_positive & recommended)
        total_recommended_relevant += hits_this_user
        total_relevant += len(actual_positive)
    
    # Recall: of everything the user actually liked, what fraction did we recommend?
    recall = total_recommended_relevant / total_relevant if total_relevant > 0 else 0
    
    # Precision: of everything we recommended, what fraction did users actually like?
    num_users = test_df['user_id'].nunique()
    precision = total_recommended_relevant / (num_users * k)
    
    return precision, recall

precision, recall = precision_recall_at_k(test, top_k_popular, K)
print(f"Recall@{K}: {recall:.4f}")
print(f"Precision@{K}: {precision:.4f}")


[20, 2, 92, 2376, 99, 726, 100, 1147, 1160, 1167]
Recall@10: 0.0673
Precision@10: 0.0678


In [ ]:
anime_ids = train['anime_id'].astype('category')
anime_id_map = dict(enumerate(anime_ids.cat.categories))      
anime_id_map_reverse = {v: k for k, v in anime_id_map.items()} 

user_ids = train['user_id'].astype('category')
user_id_map = dict(enumerate(user_ids.cat.categories))
user_id_map_reverse = {v: k for k, v in user_id_map.items()}

train['anime_idx'] = anime_ids.cat.codes
train['user_idx'] = user_ids.cat.codes

test['anime_idx'] = anime_ids.cat.codes
test['user_idx'] = user_ids.cat.codes

p_train = train[train['is_positive']==1]

item_user_matrix = csr_matrix(
    (
    [1] * len(p_train), (p_train['anime_idx'], p_train['user_idx'])
    ),
    shape = (len(anime_id_map), len(user_id_map))
)

item_similarity = cosine_similarity(item_user_matrix, dense_output=False)
save_npz(os.path.join(DATA_DIR, "item_user_matrix.npz"), item_user_matrix)
save_npz(os.path.join(DATA_DIR, "item_similarity.npz"), item_similarity)

In [ ]:
test['anime_idx'] = anime_ids.cat.codes
test['user_idx'] = user_ids.cat.codes

In [29]:
def recommend_for_user(user_idx, k=10):
    # Get this user's positively-rated anime (as matrix indices)
    user_rated = p_train[p_train['user_idx'] == user_idx]['anime_idx'].tolist()
    
    if not user_rated:
        return []  
    

    scores = np.asarray(item_similarity[user_rated].sum(axis=0)).flatten()
    
    scores[user_rated] = -1
    
    top_idx = scores.argsort()[::-1][:k]
    
    return [anime_id_map[i] for i in top_idx]

sample_user_idx = 5
recs = recommend_for_user(sample_user_idx, k=10)

path = kagglehub.dataset_download("ramazanturann/user-animelist-dataset")
animes = pd.read_csv(os.path.join(path, "animes.csv"))

print(f'10 Recommend animes: \n\n {animes[animes['animeID'].isin(recs)][['animeID', 'title']]}')

10 Recommend animes: 

      animeID                                    title
11        12                             Cowboy Bebop
37        38                         Samurai Champloo
112      113                                Fate/Zero
114      115                           Bakemonogatari
147      148  Code Geass: Lelouch of the Rebellion R2
222      223                  Neon Genesis Evangelion
282      283                    Great Teacher Onizuka
399      400                            Gurren Lagann
410      411                                 Baccano!
717      718                       Fate/Zero Season 2


In [33]:
print(train)

           user_id  anime_id  rating  is_positive  anime_idx  user_idx
143744527  1725140       403       6            0        402   1724678
40210946    765791      4724       8            1       4723    765699
43655901    793347       833       6            0        832    793240
56889005    899914       104       9            1        103    899746
143581245  1723218      1987       7            0       1986   1722756
...            ...       ...     ...          ...        ...       ...
35802234    727783      7363      10            1       7362    727714
133410375  1609156      5296       7            0       5295   1608696
147719157  1769768       316       9            1        315   1769305
127708200  1545759       163       6            0        162   1545302
56814816    899268      5693       7            0       5692    899100

[118387327 rows x 6 columns]


In [ ]:
def precision_recall_itemcf(test_df, k=10, sample_users=None):
    total_relevant = 0
    total_recommended_relevant = 0
    
    test_positive = test_df[test_df['is_positive'] == 1]
    grouped = test_positive.groupby('user_idx')
    
    users_to_eval = list(grouped.groups.keys())
    if sample_users:
        users_to_eval = np.random.choice(users_to_eval, size=sample_users, replace=False)
    
    for user_idx in users_to_eval:
        group = grouped.get_group(user_idx)
        actual_positive = set(group['anime_id'])
        
        recommended = set(recommend_for_user(user_idx, k=k))  # ← computed fresh, per user
        
        hits_this_user = len(actual_positive & recommended)
        total_recommended_relevant += hits_this_user
        total_relevant += len(actual_positive)
    
    recall = total_recommended_relevant / total_relevant if total_relevant > 0 else 0
    precision = total_recommended_relevant / (len(users_to_eval) * k)
    
    return precision, recall

precision_recall_itemcf(test, k=10, sample_users=10)

NameError: name 'test' is not defined